In [9]:

!pip install -q --upgrade kaggle
%pip install kagglehub

  Using cached kagglehub-1.0.3-py3-none-any.whl.metadata (40 kB)
  Using cached certifi-2026.10.23-py3-none-any.whl.metadata (2.5 kB)
  Using cached kagglesdk-0.1.38-py3-none-any.whl.metadata (13 kB)
Using cached kagglehub-1.0.3-py3-none-any.whl (73 kB)
Using cached kagglesdk-0.1.38-py3-none-any.whl (290 kB)
Using cached certifi-2026.10.23-py3-none-any.whl (135 kB)

  Attempting uninstall: certifi

    Found existing installation: certifi 2026.7.22

    Uninstalling certifi-2026.7.22:

      Successfully uninstalled certifi-2026.7.22

   ------------- -------------------------- 1/3 [kagglesdk]
   ------------- -------------------------- 1/3 [kagglesdk]
   ------------- -------------------------- 1/3 [kagglesdk]
   ------------- -------------------------- 1/3 [kagglesdk]
   ------------- -------------------------- 1/3 [kagglesdk]
   ------------- -------------------------- 1/3 [kagglesdk]
   -------------------------- ------------- 2/3 [kagglehub]
   -------------------------- ---------

In [10]:
import kagglehub
kagglehub.login()

In [11]:
import kagglehub

# Download latest version
path = kagglehub.competition_download("who-is-who-in-text-generation")

print("Path to competition files:", path)

100%|██████████| 214k/214k [00:00<00:00, 911kB/s]

Extracting files...
Path to competition files: C:\Users\guill\.cache\kagglehub\competitions\who-is-who-in-text-generation


In [12]:
import os

for root, dirs, files in os.walk(path):
    for f in files:
        print(os.path.join(root, f))

C:\Users\guill\.cache\kagglehub\competitions\who-is-who-in-text-generation\sample_submission.csv
C:\Users\guill\.cache\kagglehub\competitions\who-is-who-in-text-generation\test.csv
C:\Users\guill\.cache\kagglehub\competitions\who-is-who-in-text-generation\train.csv


In [16]:
import pandas as pd
from pathlib import Path

# Ajusta el nom segons els fitxers que hagi llistat la cel·la anterior
train = pd.read_csv(Path(path) / "train.csv")
train.head(10)

,id,age,sex,bmi,smoking_status,encounter_type,systolic_bp,diastolic_bp,heart_rate,respiratory_rate,...,diabetes,hypertension,chronic_kidney_disease,primary_diagnosis,num_medications,prior_visits_12m,length_of_stay_days,discharge_destination,readmitted_30d,label
0,5be091fe7010722968d37fbbbfa5adec,38,Male,30.4,Current,Urgent,160,95,105,28,...,No,No,No,Respiratory,6,10,3.0,Home,No,Model_B
1,7ebc8ab8f1dfede1237163cb2e71325f,48,Female,28.5,Current,Elective,128,76,78,16,...,No,No,No,Gastrointestinal,3,2,2.0,Home,No,Model_D
2,69f51e35fec3bda1d81f2cbf9ac8b827,53,Female,25.4,Never,Elective,130,78,77,19,...,No,No,No,Renal,1,1,3.0,Rehabilitation,No,Model_A
3,8c64aa346b72cad6cb179f254abfd71d,45,Male,28.2,Never,Elective,145,95,72,18,...,Yes,No,No,Cardiovascular,4,0,4.0,Home,No,Model_C
4,c8255d88bcfc566c5616bed411b55bd7,41,Male,27.9,Current,Emergency,150,94,115,22,...,No,No,No,Gastrointestinal,2,5,3.0,HomeHealth,No,Model_B
5,b9eb25a34b7bfd499053b23811f887c1,51,Female,NaN,Former,Outpatient,140,85,107,20,...,No,No,Yes,Cardiovascular,6,2,1.0,Home,No,Model_B
6,1afa743a49eedc367a161f8712c1478f,35,Male,30.4,Current,Elective,142,90,100,22,...,No,Yes,No,Endocrine,5,6,2.0,Home,No,Model_B
7,fffd77125edca073a39e521d02cc7540,60,Male,27.8,Never,Outpatient,130,85,70,12,...,No,Yes,No,Infectious,0,1,1.0,Home,No,Model_C
8,e410df72be53e83100a5279bcd87aaff,42,Female,23.1,Former,Outpatient,120,84,74,16,...,No,No,No,Other,2,0,1.0,Home,No,Model_A
9,f22756a4d16fcb7fe22266b1a53121e0,55,Female,24.3,Current,Urgent,133,82,79,16,...,No,No,No,Cardiovascular,5,3,0.0,Home,No,Model_D


In [17]:
#MODEL 3 — MODEL 3

import numpy as np
import pandas as pd
from pathlib import Path
import kagglehub
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, KBinsDiscretizer, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

# Data
path = kagglehub.competition_download("who-is-who-in-text-generation")
train = pd.read_csv(Path(path) / "train.csv")
ID = "id"
TARGET = "label" if "label" in train.columns else train.columns[-1]

raw = train.drop(columns=[ID, TARGET]) # Raw features, before add_features
y = train[TARGET]
ORIG_NUM = raw.select_dtypes("number").columns.tolist()  # Original numeric columns
DECIMAL_COLS = [c for c in ORIG_NUM if (raw[c].dropna() != raw[c].dropna().round(0)).any()] # Columns with decimals


# Testing diferent features
ACTIVE_GROUPS = ["n_missing", "mult7", "mult5", "mult10", "decimals", "digits", "values", "squares", "bins", "interactions"] # This is for adding or removing possible features (now it is cleaned up to the ones used in the final model)
USE_OVR = True
MIN_FREQ_VALUES = 10 # For the frequency function further below

# Functions
def n_decimals(s): # Number of decimals of each value
    out = pd.Series(np.nan, index=s.index)
    for k in [4, 3, 2, 1, 0]:
        out = out.mask((s - s.round(k)).abs() < 1e-9, k)
    return out

def pairwise_products(Z): # z_i * z_j for every pair of standardized columns
    Z = np.asarray(Z, dtype=float)
    i, j = np.triu_indices(Z.shape[1], k=1)
    return Z[:, i] * Z[:, j]

# Adding features
def add_features(df, groups=None):
    if groups is None:
        groups = ACTIVE_GROUPS # The groups for testing (this is the final decision os new features)
    df = df.copy()
    num_cols = [c for c in ORIG_NUM if c in df.columns]
    if "n_missing" in groups:
        df["n_missing"] = df.isna().sum(axis=1) # Number of NaN in the row
    for c in num_cols:
        s = df[c]
        if "mult7" in groups:
            df[f"{c}__mult7"]  = ((s % 7) == 0).astype(int) # 1 if multiple of 7
        if "mult5" in groups:
            df[f"{c}__mult5"]  = ((s % 5) == 0).astype(int) # 1 if multiple of 5
        if "mult10" in groups:
            df[f"{c}__mult10"] = ((s % 10) == 0).astype(int) # 1 if multiple of 10
        if "decimals" in groups:
            df[f"{c}__n_dec"]  = n_decimals(s) # Number of decimals used
        if "digits" in groups: # Favorite final digits, as categories. OHE in the text lane
            df[f"{c}__last_digit"] = (s.abs() // 1 % 10).map(lambda v: "NA" if pd.isna(v) else f"d{int(v)}")
            if c in DECIMAL_COLS:
                df[f"{c}__first_decimal"] = ((s.abs() * 10).round() % 10).map(lambda v: "NA" if pd.isna(v) else f"d{int(v)}")
    return df

# Model
def make_model(X, groups=None, ovr=None):
    if groups is None:
        groups = ACTIVE_GROUPS
    if ovr is None:
        ovr = USE_OVR
    num_cols  = X.select_dtypes("number").columns.tolist()
    text_cols = X.select_dtypes(exclude="number").columns.tolist()

    lanes = [
        ("number", Pipeline([
            ("imput median", SimpleImputer(strategy="median", add_indicator=True)), # Median + "missing" columns
            ("normalize",  StandardScaler()), # Standarization
        ]), num_cols),
        ("text", Pipeline([
            ("missing filler", SimpleImputer(strategy="constant", fill_value="__missing__")),
            ("one hot encoding", OneHotEncoder(handle_unknown="ignore")), # OHE
        ]), text_cols),
    ]
    if "values" in groups: # Saving repeating values
        lanes.append(("values", Pipeline([
            ("fill", SimpleImputer(strategy="constant", fill_value=-999)),
            ("ohe",  OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=MIN_FREQ_VALUES)),
        ]), ORIG_NUM))
    if "squares" in groups: # Variance of data (squared z-score)
        lanes.append(("squares", Pipeline([
            ("fill", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
            ("square", FunctionTransformer(np.square)),
            ("rescale", StandardScaler()),
        ]), ORIG_NUM))
    if "bins" in groups: # Ranges of the numbers (quantiles)
        lanes.append(("bins", Pipeline([
            ("fill", SimpleImputer(strategy="median")),
            ("kbins", KBinsDiscretizer(n_bins=8, encode="onehot", strategy="quantile")),
        ]), ORIG_NUM))
    if "interactions" in groups: # Relationships between variables: z_i * z_j (LLM likes high or low numbers in some features)
        lanes.append(("interactions", Pipeline([
            ("fill", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
            ("products", FunctionTransformer(pairwise_products)),
            ("rescale", StandardScaler()),
        ]), ORIG_NUM))

    clf = LogisticRegression(penalty=None, max_iter=10000) # C=0
    if ovr:
        clf = OneVsRestClassifier(clf) # Default parameters
    return Pipeline([("Preporcess", ColumnTransformer(lanes)), ("LogiReg", clf)])

# Cross validation
cv = StratifiedKFold(n_splits=20, shuffle=True, random_state=590)
X = add_features(raw)
model = make_model(X)
cv_pred = cross_val_predict(model, X, y, cv=cv, n_jobs=-1)
cv_acc = accuracy_score(y, cv_pred)  # Accuracy with cross validation

model.fit(X, y) # Training with all the labeled data
train_pred = model.predict(X)
train_acc = accuracy_score(y, train_pred)
n_columns = model[:-1].transform(X).shape[1]
MODEL_NAME = "MODEL 3 (best)"
read_data = pd.read_csv               

print(f"{MODEL_NAME} | groups: {ACTIVE_GROUPS} | {n_columns} columns")
print(f"Accuracy en TRAIN (optimistic):        {train_acc:.2%}")
print(f"Accuracy en VALIDACIÓN CRUZADA (real): {cv_acc:.2%}\n")
print(classification_report(y, cv_pred, digits=3))
print("Matriz de confusión (CV):")
display(pd.DataFrame(confusion_matrix(y, cv_pred, labels=model.classes_),
                     index=model.classes_, columns=model.classes_))

pre = model.named_steps["Preporcess"]
total = 0
for name, sl in pre.output_indices_.items():
    n = sl.stop - sl.start
    total += n
    print(f"{name:15s} {n}")
print("TOTAL", total)

MODEL 3 (best) | groups: ['n_missing', 'mult7', 'mult5', 'mult10', 'decimals', 'digits', 'values', 'squares', 'bins', 'interactions'] | 1035 columns
Accuracy en TRAIN (optimistic):        100.00%
Accuracy en VALIDACIÓN CRUZADA (real): 93.04%

              precision    recall  f1-score   support

     Model_A      0.932     0.957     0.944       600
     Model_B      0.936     0.920     0.928       600
     Model_C      0.919     0.922     0.920       600
     Model_D      0.936     0.923     0.930       600

    accuracy                          0.930      2400
   macro avg      0.930     0.930     0.930      2400
weighted avg      0.930     0.930     0.930      2400

Matriz de confusión (CV):


,Model_A,Model_B,Model_C,Model_D
Model_A,574,7,9,10
Model_B,16,552,21,11
Model_C,9,21,553,17
Model_D,17,10,19,554


number          86
text            226
values          494
squares         15
bins            109
interactions    105
remainder       0
TOTAL 1035


In [ ]:
COMP = "who-is-who-in-text-generation"
MSG  = f"LogReg C=np.inf"   # anota el CV para saber qué era cada envío
#LO DE ABAJO ES PARA HACER SUBMIT OJO OJO OJO OJO, descomentar para submittear
#!kaggle competitions submit -c {COMP} -f submission.csv -m "{MSG}"

In [ ]:
#PseudoCode
'''
PROGRAM  "MODEL 3 (best)": guess which of 4 LLMs (Model_A…Model_D) generated each patient record

── LOAD DATA ─────────────────────────────────────────────────────────────
download the competition files
train   ← read "train.csv"                       # 2400 rows
y       ← train["label"]                         # the answer: Model_A, B, C or D
raw     ← train without columns "id" and "label" # the clinical variables
ORIG_NUM     ← names of the numeric columns in raw
DECIMAL_COLS ← numeric columns that ever contain a non-integer value

ACTIVE_GROUPS ← [n_missing, mult7, mult5, mult10, decimals, digits,   # made by add_features
                 values, squares, bins, interactions]                  # made inside the pipeline
USE_OVR ← TRUE
MIN_FREQ_VALUES ← 10

── FUNCTION add_features(table) ─────────────────────────────────────────
copy the table
IF n_missing: new column = number of empty cells in the row
FOR each original numeric column c:
    IF mult7:    new column = 1 if value is a multiple of 7, else 0
    IF mult5:    new column = 1 if value is a multiple of 5, else 0
    IF mult10:   new column = 1 if value is a multiple of 10, else 0
    IF decimals: new column = number of decimals of the value (smallest k with round(x,k) = x)
    IF digits:   new TEXT column = last digit of the integer part   ("d0"…"d9")
                 IF c has decimals: new TEXT column = first decimal digit
RETURN the extended table

── FUNCTION make_model(X) ───────────────────────────────────────────────
num_cols  ← numeric columns of X   (original + created)
text_cols ← text columns of X      (original categories + digit columns)

build a ColumnTransformer with "lanes" (each lane = columns + transformations):
  lane "number":       num_cols   → fill gaps with median (+ add "was missing" 0/1 columns) → standardize
  lane "text":         text_cols  → fill gaps with "__missing__" → one-hot
  lane "values":       ORIG_NUM   → fill gaps with -999 → one-hot of each exact value
                                    (only values seen ≥ 10 times get their own column; rest → "infrequent")
  lane "squares":      ORIG_NUM   → fill median → standardize → square each value → standardize
  lane "bins":         ORIG_NUM   → fill median → split into 8 quantile ranges → one-hot
  lane "interactions": ORIG_NUM   → fill median → standardize → product of every pair (z_i · z_j) → standardize
  (all lanes' outputs are placed side by side → one big numeric matrix)

classifier ← LogisticRegression(penalty = None, max_iter = 10000)
IF USE_OVR: classifier ← OneVsRestClassifier(classifier)     # 4 binary models: "A vs rest", "B vs rest", …
RETURN Pipeline( ColumnTransformer → classifier )

── EVALUATE AND TRAIN ───────────────────────────────────────────────────
X     ← add_features(raw)
model ← make_model(X)

cv ← 20 stratified folds (shuffled, seed 590)
FOR each fold:
    fit a fresh copy of model on the other 19 folds   # medians, scales, categories learned ONLY there
    predict the held-out fold
cv_pred ← all held-out predictions
cv_acc  ← % of cv_pred equal to y                     # honest estimate

model.fit(X, y)                                       # final model, trained on all 2400 rows
train_acc ← % correct when predicting X itself        # optimistic (model has seen these rows)
n_columns ← width of the matrix after preprocessing

PRINT name, groups, n_columns, train_acc, cv_acc
PRINT precision / recall / F1 per class (from cv_pred)
PRINT confusion matrix (real class × predicted class)
'''

In [ ]:
#Pseudoer Pseudocode
'''
STEP 1 – Look at the records.
         For each record we know who wrote it (the "answer key").

STEP 2 – Look for each AI's "habits" (its fingerprint):
         • Does it leave fields blank?
         • Does it like round numbers (ending in 0 or 5) or the number 7?
         • Which final digits does it prefer?
         • How many decimals does it use?
         • Does it repeat the same "favourite" values (e.g. always blood pressure 120)?
         • Are its values usually typical or extreme?
         • Does it make values go together in a particular way
           (e.g. high blood pressure together with high heart rate)?

STEP 3 – Turn all those habits into a long list of YES/NO answers and numbers
         for every record, so a computer can compare them.

STEP 4 – Give that list to a simple "scoring" method (logistic regression).
         It learns how many points each habit gives to each AI.
         For a new record it adds up the points, and the AI with the most points wins.

STEP 5 – Check it honestly:
         Split the records into 20 groups. Learn from 19, test on the remaining one,
         and repeat 20 times. The percentage of correct guesses on records it had
         NEVER seen = the real accuracy (about 93%).

STEP 6 – Finally, learn from all 2400 records together.
         That final version is the one used to guess the authors of the test records.
'''